In [16]:
import numpy as np
import scanpy as sc
import pandas as pd
# import cosg
import pandas as pd
import numpy as np
import time
import os

def ssim(part1, part2):
    # Replace NaN values with 0
    mu1 = np.mean(part1)
    mu2 = np.mean(part2)
    
    C1 = 0.01**2
    C2 = 0.03**2
    
    numerator = (2 * mu1 * mu2 + C1) * (2 * np.cov(part1, part2)[0, 1] + C2)
    denominator = (mu1**2 + mu2**2 + C1) * (np.var(part1) + np.var(part2) + C2)
    
    ssim_result = numerator / denominator
    return ssim_result

def KL(part1, part2):
    ee=0.001
    ratio = (part1+ee) / (part2+ee)
    result = part1 * np.log(ratio)
    
    # Avoiding extreme values
    result[result > 1000] = 0
    result[result < (-1000)] = 0
    
    return np.sum(result)

def JS(part1, part2):
    M = (part1 + part2) / 2
    return 0.5 * KL(part1, M) + 0.5 * KL(part2, M)

def cosi(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

def pear(vec1, vec2):
    return np.corrcoef(vec1, vec2)[0, 1]

def mse(vec1, vec2):
    return np.mean((vec1 - vec2) ** 2)
def common_gene(sc_data,st_data,anno):
    
    # Retrieve the gene lists for sc_data and st_data.
    sc_genes = sc_data.var_names
    st_genes = st_data.var_names
    # Use set operations to find common genes.
    common_genes = set(sc_genes).intersection(set(st_genes))
    # Retain only the genes common to both sc_data and st_data.
    sc_data = sc_data[:, list(common_genes)]
    st_data = st_data[:, list(common_genes)]

    marker_genes = pd.read_csv('/data/work/subclassMarke.tsv', delimiter='\t')  ###Marker Genes for Each Cell Type in snRNA-seq
    top_genes = marker_genes.groupby('cluster').apply(lambda x: x.nlargest(300, 'avg_log2FC')).reset_index(drop=True)
    intersect_genes = st_data.var_names.intersection(top_genes['gene'])
    sc_data = sc_data[:, list(intersect_genes)]
    st_data = st_data[:, list(intersect_genes)]
    sorted_genes = sorted(st_data.var_names)

    st_data = st_data[:, sorted_genes]
    sc_data = sc_data[:, sorted_genes]
    
    df_grouped_means = pd.DataFrame(index=sc_data.obs[anno].cat.categories, columns=sc_data.var_names)
    # For each annotation category, calculate the average gene expression.
    for annotation in sc_data.obs[anno].unique():
        subset = sc_data[sc_data.obs[anno] == annotation, :]
        mean_expression = np.mean(subset.X, axis=0)
        df_grouped_means.loc[annotation] = mean_expression
        
    return sc_data,st_data,df_grouped_means

In [25]:
Chip = "GroupI-1"

In [26]:
st_obj = sc.read_h5ad("/data/"+Chip+"_addmeta.h5ad")
sc_obj=sc.read_h5ad("/data/snRNA_downsample.h5ad")
sc_obj.var_names_make_unique()

In [27]:
st_data=st_obj
sc_data=sc_obj
anno = 'subclass.v4'
sc_data,st_data,df_grouped_means = common_gene(sc_data,st_data,anno)

In [28]:
import os

Method_list = ["CellType_Spatialid", "CellType_Tangram", "CellType_Cell2", "CellType_Destvi", 
               "CellType_RCTD", "CellType_Spotlight", "CellType_Seurat", "CellType_Spann"]

all_results = []

# Iterate over the methods
for method in Method_list:
    st_data.obs = st_data.obs.copy()
    st_data.obs["annotation"] = st_data.obs[method]

    # Sample fraction of cells (adjust as needed)
    sample_fraction = 0.25
    groups = st_data.obs.groupby('annotation')
    sampled_cells = groups.apply(lambda x: x.sample(frac=sample_fraction, random_state=np.random.RandomState(42)))

    # List to store rows before creating DataFrame
    results_list = []

    for cell in sampled_cells.index.get_level_values(1):  # Iterate over sampled cells
        cell_type = st_data.obs.loc[cell, 'annotation']
        real_expression = st_data[cell, :].X.toarray().squeeze()

        # Initialize the row dictionary
        new_row = {
            "Cell": cell,
            "Type": cell_type,
            "Method": method  # Add the method column
        }

        if cell_type != "unknown":
            if cell_type in df_grouped_means.index:
                avg_expression = df_grouped_means.loc[cell_type].values
                real_expression = real_expression.astype('float')
                avg_expression = avg_expression.astype('float')

                # Calculate metrics
                new_row.update({
                    "SSIM": ssim(real_expression, avg_expression),
                    "KL": KL(real_expression, avg_expression),
                    "JS": JS(real_expression, avg_expression),
                    "COSINE": cosi(real_expression, avg_expression),
                    "PEARSON": pear(real_expression, avg_expression),
                    "MSE": mse(real_expression, avg_expression)
                })
            else:
                # If cell type not found in df_grouped_means
                new_row.update({
                    "SSIM": 0,
                    "KL": 1000,
                    "JS": 1000,
                    "COSINE": 0,
                    "PEARSON": 0,
                    "MSE": 1000
                })
        else:
            # If the cell type is "unknown"
            new_row.update({
                "SSIM": 0,
                "KL": 1000,
                "JS": 1000,
                "COSINE": 0,
                "PEARSON": 0,
                "MSE": 1000
            })

        # Append the new row to the list
        results_list.append(new_row)

    # Create the DataFrame once after the loop
    results_df = pd.DataFrame(results_list)

    # Append the results of this method to the all_results list
    all_results.append(results_df)

# Concatenate all results into a single DataFrame
combined_results = pd.concat(all_results, ignore_index=True)

# Ensure output directories exist
output_dir_all = "/data/ACC/All/"
output_dir_avg = "/data/ACC/average/"
os.makedirs(output_dir_all, exist_ok=True)
os.makedirs(output_dir_avg, exist_ok=True)

# Save the combined results DataFrame
combined_results.to_csv(os.path.join(output_dir_all, f"{Chip}_ACC_all_methods_cell_values.csv"), index=False)

mean_values_per_method = combined_results.groupby('Method').agg({
    "SSIM": "mean",
    "KL": "mean",
    "JS": "mean",
    "COSINE": "mean",
    "PEARSON": "mean",
    "MSE": "mean"
}).reset_index()

# Save mean values per method to a DataFrame and then to CSV
mean_values_per_method.to_csv(os.path.join(output_dir_avg, f"{Chip}_ACC_all_methods_average_values_per_method.csv"), index=False)

In [30]:
mean_values_per_method

,Method,SSIM,KL,JS,COSINE,PEARSON,MSE
0,CellType_Cell2,0.202781,283.649413,404.987222,0.488954,0.465744,1.139475
1,CellType_Destvi,0.201431,248.312319,416.184531,0.511493,0.490048,1.287681
2,CellType_RCTD,0.179844,208.096229,446.654491,0.529110,0.509433,1.699642
3,CellType_Seurat,0.190353,243.570546,439.237012,0.507679,0.487016,15.220470
4,CellType_Spann,0.142605,520.957700,562.764037,0.321087,0.299289,235.810271
5,CellType_Spatialid,0.202921,321.312879,410.131874,0.460250,0.434567,0.965215
6,CellType_Spotlight,0.181176,219.016996,441.391973,0.523030,0.502919,1.682144
7,CellType_Tangram,0.192363,324.228304,425.876431,0.440097,0.413701,1.223129


In [7]:
# import os

# Method_list = ["CellType_Spatialid", "CellType_Tangram", "CellType_Cell2", "CellType_Destvi", 
#                "CellType_RCTD", "CellType_Spotlight", "CellType_Seurat", "CellType_Spann"]

# # Define Chip or get from another context
# Chip = "example_chip"  # Replace with your actual Chip value or pass it into the function

# # Initialize an empty list to store the combined results
# all_results = []

# # Iterate over the methods
# for method in Method_list:
#     st_data.obs = st_data.obs.copy()
#     st_data.obs["annotation"] = st_data.obs[method]

#     # Sample fraction of cells (adjust as needed)
#     sample_fraction = 0.001
#     groups = st_data.obs.groupby('annotation')
#     sampled_cells = groups.apply(lambda x: x.sample(frac=sample_fraction, random_state=np.random.RandomState(42)))

#     # List to store rows before creating DataFrame
#     results_list = []

#     for cell in sampled_cells.index.get_level_values(1):  # Iterate over sampled cells
#         cell_type = st_data.obs.loc[cell, 'annotation']
#         real_expression = st_data[cell, :].X.toarray().squeeze()

#         # Initialize the row dictionary
#         new_row = {
#             "Cell": cell,
#             "Type": cell_type,
#             "Method": method  # Add the method column
#         }

#         if cell_type != "unknown":
#             if cell_type in df_grouped_means.index:
#                 avg_expression = df_grouped_means.loc[cell_type].values
#                 real_expression = real_expression.astype('float')
#                 avg_expression = avg_expression.astype('float')

#                 # Calculate metrics
#                 new_row.update({
#                     "SSIM": ssim(real_expression, avg_expression),
#                     "KL": KL(real_expression, avg_expression),
#                     "JS": JS(real_expression, avg_expression),
#                     "COSINE": cosi(real_expression, avg_expression),
#                     "PEARSON": pear(real_expression, avg_expression),
#                     "MSE": mse(real_expression, avg_expression)
#                 })
#             else:
#                 # If cell type not found in df_grouped_means
#                 new_row.update({
#                     "SSIM": 0,
#                     "KL": 1000,
#                     "JS": 1000,
#                     "COSINE": 0,
#                     "PEARSON": 0,
#                     "MSE": 1000
#                 })
#         else:
#             # If the cell type is "unknown"
#             new_row.update({
#                 "SSIM": 0,
#                 "KL": 1000,
#                 "JS": 1000,
#                 "COSINE": 0,
#                 "PEARSON": 0,
#                 "MSE": 1000
#             })

#         # Append the new row to the list
#         results_list.append(new_row)

#     # Create the DataFrame once after the loop
#     results_df = pd.DataFrame(results_list)

#     # Append the results of this method to the all_results list
#     all_results.append(results_df)

# # Concatenate all results into a single DataFrame
# combined_results = pd.concat(all_results, ignore_index=True)

# # Ensure output directories exist
# output_dir_all = "/data/work/Layer division/01_Result/ACC/All/"
# output_dir_avg = "/data/work/Layer division/01_Result/ACC/average/"
# os.makedirs(output_dir_all, exist_ok=True)
# os.makedirs(output_dir_avg, exist_ok=True)

# # Save the combined results DataFrame
# combined_results.to_csv(os.path.join(output_dir_all, f"{Chip}_ACC_all_methods_cell_values.csv"), index=False)

# # Calculate mean values for each metric
# mean_values = {
#     "SSIM": combined_results["SSIM"].mean(),
#     "KL": combined_results["KL"].mean(),
#     "JS": combined_results["JS"].mean(),
#     "COSINE": combined_results["COSINE"].mean(),
#     "PEARSON": combined_results["PEARSON"].mean(),
#     "MSE": combined_results["MSE"].mean()
# }

# # Save mean values to a DataFrame and then to CSV
# mean_df = pd.DataFrame.from_dict(mean_values, orient='index', columns=['Average Value'])
# mean_df.to_csv(os.path.join(output_dir_avg, f"{Chip}_ACC_all_methods_average_values.csv"))
